In [10]:
import os
import angr
import magic

SEARCH_LENGTH = 2048


def get_file_kinds(path):
    files = os.listdir(path)
    kinds = {}
    for file in files:
        full_path = os.path.join(path, file)
        kind = magic.from_buffer(open(full_path, "rb").read(SEARCH_LENGTH))
        kinds[full_path] = kind
    return kinds


MAL_PATH = "/Volumes/New Volume/malware-detection-dataset/malware"
BEN_PATH = "/Volumes/New Volume/malware-detection-dataset/benign"

WARNING  | 2025-01-28 09:04:44,926 | angr.state_plugins.unicorn_engine | Unicorn is not installed. Support disabled.
WARNING  | 2025-01-28 09:04:44,938 | angr.state_plugins.unicorn_engine | failed loading "angr_native.dylib", unicorn support disabled ('NoneType' object has no attribute 'unicorn')


In [28]:
mal_kinds = get_file_kinds(MAL_PATH)
ben_kinds = get_file_kinds(BEN_PATH)

In [13]:
from collections import Counter

Counter(mal_kinds.values()).most_common(10)

[('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 4 sections', 650),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 3 sections', 599),
 ('PE32 executable for MS Windows 5.01 (GUI), Intel i386, 5 sections', 555),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 5 sections', 502),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, UPX compressed, 3 sections',
  254),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 8 sections', 227),
 ('PE32 executable for MS Windows 5.00 (GUI), Intel i386, 5 sections', 178),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 7 sections', 146),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 6 sections', 138),
 ('PE32 executable for MS Windows 4.00 (console), Intel i386, 4 sections',
  130)]

In [14]:
Counter(ben_kinds.values()).most_common(10)

[('data', 667),
 ('PE32+ executable for MS Windows 6.00 (console), x86-64, 5 sections', 447),
 ('PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 9 sections',
  370),
 ('PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 10 sections',
  299),
 ('PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 11 sections',
  253),
 ('PE32 executable for MS Windows 4.00 (console), Intel i386 (stripped to external PDB), 2 sections',
  228),
 ('PE32 executable for MS Windows 4.00 (console), Intel i386 (stripped to external PDB), 10 sections',
  219),
 ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 5 sections', 179),
 ('PE32+ executable for MS Windows 5.02 (GUI), x86-64, 11 sections', 175),
 ('PE32 executable for MS Windows 5.01 (console), Intel i386, 4 sections',
  160)]

In [29]:
Counter(ben_kinds.values())

Counter({'data': 667,
         'PE32+ executable for MS Windows 6.00 (console), x86-64, 5 sections': 447,
         'PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 9 sections': 370,
         'PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 10 sections': 299,
         'PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 11 sections': 253,
         'PE32 executable for MS Windows 4.00 (console), Intel i386 (stripped to external PDB), 2 sections': 228,
         'PE32 executable for MS Windows 4.00 (console), Intel i386 (stripped to external PDB), 10 sections': 219,
         'PE32 executable for MS Windows 4.00 (GUI), Intel i386, 5 sections': 179,
         'PE32+ executable for MS Windows 5.02 (GUI), x86-64, 11 sections': 175,
         'PE32 executable for MS Windows 5.01 (console), Intel i386, 4 sections': 160,
         'PE32 executable for MS Windows 4.00 (console), Intel i386 Mono/.Net a

In [26]:
len([file for file, kind in ben_kinds.items() if kind.startswith("PE32 executable")])

2326

In [16]:
import numpy as np


mal = np.array(list(mal_kinds.keys()))
labels = np.zeros_like(list(mal_kinds.keys()))
labels[:] = 1

mal_dset = np.vstack([mal, labels])

In [17]:
ben = np.array(list(ben_kinds.keys()))
labels = np.zeros_like(list(ben_kinds.keys()))
labels[:] = 0

ben_dset = np.vstack([ben, labels])

In [18]:
import pandas as pd

dset = np.hstack([mal_dset, ben_dset])
dset = pd.DataFrame(dset.T)

path, label = dset.iloc[0, :]
path, label

('/Volumes/New Volume/malware-detection-dataset/malware/VirusShare_39edbfc07c389a37b11948e029c297c1',
 '1')